# InferScale-X · 03 Alternative engines + NVIDIA Dynamo attempt
Each section needs a **fresh runtime** (Runtime → Disconnect and delete runtime), then re-run section 0. Results accumulate in Drive via `save_results.sh`; compare at the end.

## 0 · Get the code
Upload `InferScale-X.zip` to your Drive root (or to this session), then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/InferScale-X'):
    !cp /content/drive/MyDrive/InferScale-X.zip /content/ 2>/dev/null || echo 'upload InferScale-X.zip to /content'
    !cd /content && unzip -q -o InferScale-X.zip
%cd /content/InferScale-X
!nvidia-smi

## A · SGLang

In [ ]:
!bash scripts/colab_setup.sh sglang
!python inference/exp_engines.py --engine sglang --repeats 3

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## B · TensorRT-LLM (experimental; fresh runtime)

In [ ]:
%cd /content/InferScale-X
# TensorRT-LLM ships no Python 3.13 wheels (PyPI has only an sdist that cannot build here), so it runs in a
# Python 3.12 venv. trtllm-serve lives in the venv; the load generator runs in Colab's own Python.
!pip install -q uv && pip install -q -r requirements/dev.txt
!uv venv /content/trt --python 3.12 -q
!uv pip install -q --python /content/trt/bin/python tensorrt_llm --extra-index-url https://pypi.nvidia.com
!/content/trt/bin/python -c "import tensorrt_llm as t; print('tensorrt_llm', t.__version__)"
!PATH=/content/trt/bin:$PATH python inference/exp_engines.py --engine trtllm --repeats 3


In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## C · NVIDIA Dynamo single-node attempt (fresh runtime). `status.json` says whether it actually served.

In [ ]:
!pip install -q -r requirements/dev.txt
!bash inference/dynamo/try_dynamo.sh

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## D · Compare (after copying every runtime's results back from Drive)

In [ ]:
!cp -rn /content/drive/MyDrive/InferScale-X-results/results/* benchmarks/results/ 2>/dev/null; python benchmarks/analysis/compare_engines.py